# Clustering III: Cluster Validation and Evaluation

Clustering always produces *some* partition of the data — the question is whether this partition is meaningful or not.

The problem is that, unlike supervised learning, we do not have anything to compare with, since clustering normally has no ground-truth labels.  

But we know that a good clustering result usually is characterized by the following properties:

- **Compactness**: points within a cluster should be close to each other.
- **Separation**: clusters should be well separated from each other.

This is what we want to assess. And we will cover several way for assessing this:

1. Visualization
2. Internal validation
3. External validation
4. Cluster stability


## 1-Validation through visualization

This is as simple as this: 

<div class="alert alert-block alert-info">
    
💡💡<b>Always</b> visualize your clustering solutions!💡💡
</div>

Let's cluster the following toy data, applying K-means searching for 3 clusters, as we know this is the real number.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
X, y = make_classification(n_samples=400, 
                           n_features=5, 
                           n_informative = 5, 
                           n_redundant=0, 
                           n_classes=3, 
                           n_clusters_per_class=1, 
                           class_sep=2.5, 
                           random_state=1234)


labels = None # HERE COMES CODE: Define KMeans with 3 clusters and some random_state. The fit and predict

Now we should visualize this clustering result. 

The problem is: how do we visualize a dataset with more than 3 dimensions (3 features)?

Let's see a few approaches for this: 

- **PCA**: We could use this dimensionality reduction by projecting our data into the first 2 components:

In [ ]:
import matplotlib.pyplot as plt
# HERE COMES CODE: import PCA 

# Project data to 2 components
pca = None # HERE COMES CODE: Define PCA with n_components=2
X_pca = None # HERE COMES CODE: transform the data to the PCA space

plt.figure(figsize=(7,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=labels, cmap='Set1',
            edgecolor='k', s=80, alpha=0.9)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Projection Colored by Cluster Labels", fontsize=15)
plt.grid(True)
plt.show()


- **Multidimensional scaling**: It also reduces the dimensionality of data, but preserving the same distance between observations points.

You can perform this using the `MDS` class in `manifold` from `scikit-learn`.

In [ ]:
# HERE COMES CODE: import MDS
mds = None # HERE COMES CODE: Define MDS with n_components=2 and some random_state
X_mds = None # HERE COMES CODE: transform the data to the MDS space

plt.figure(figsize=(7,5))
plt.scatter(X_mds[:,0], X_mds[:,1], c=labels, cmap='Set1',
            edgecolor='k', s=80, alpha=0.9)
plt.xlabel("MDS Dimension 1")
plt.ylabel("MDS Dimension 2")
plt.title("MDS Distance-Preserving Projection", fontsize=15)
plt.grid(True)
plt.show()

- **heatmap**, ordered by clustering assignment.

In [ ]:
# HERE COMES CODE: Import seaborn 
import numpy as np
import pandas as pd

df = pd.DataFrame(X)
df['cluster'] = labels

df_sorted = None # HERE COMES CODE: # Sort rows by cluster assignment
features_only = None # HERE COMES CODE: # Drop cluster column

plt.figure(figsize=(9,6))
# HERE COMES CODE: # Use heatmap from seaborn
plt.title("Cluster Heatmap (Samples Sorted by Cluster)", fontsize=15)
plt.xlabel("Features")
plt.ylabel("Samples (sorted by cluster)")
plt.show()


- **Distance matrix**, ordered by cluster assignment.

In [ ]:
# HERE COMES CODE: import pairwise_distances from metrics
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

D = None # HERE COMES CODE: Compute distance matrix 

order = # HERE COMES CODE: argsort labels
D_ordered = # Reorder D

plt.figure(figsize=(7,6))
sns.heatmap(D_ordered, cmap='viridis', square=True, cbar=True)
plt.title("Distance Matrix Ordered by Cluster", fontsize=14)
plt.xlabel("Samples (ordered by cluster)")
plt.ylabel("Samples (ordered by cluster)")
plt.show()


Again:

> **Takeaway**: always visualize your cluster solutions!

## 2- Internal Validation

Internal validation attempts to quantify the quality of a clustering just by using the data and the clustering solution.

It relies on different metrics for this.

The most common application internal validation and the following metrics is to select the right number of clusters.

### Silhouette Coefficient

For each point $i$:

- $a(i)$ = average distance to points in the same cluster  
- $b(i)$ = average distance to points in the nearest different cluster  

The silhouette score is:

$$
s(i) = \frac{b(i) - a(i)}{\max\{a(i),\, b(i)\}}
$$

As we saw in the first tutorial dedicated to clustering, this score can be calculated through `silhouette_score` in `sklearn.metrics`.

In [ ]:
# HERE COMES CODE: Import silhouette_score
import matplotlib.pyplot as plt

# Compute silhouette score for different k
K = range(2, 10)
sil_scores = []

for k in K:
    model = None # HERE COMES CODE: Define K-means with k number clusters. Set random_state to a value
    labels = None # HERE COMES CODE: Use fit_predict to predict labels
    sil = None # HERE COMES CODE: Calculate silhoutte_score
    sil_scores.append(sil)

# Plot
plt.figure(figsize=(7,5))
plt.plot(K, sil_scores, 'o-', color='steelblue')
plt.title("Silhouette Score vs Number of Clusters")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Average silhouette score")
plt.grid(True)
plt.show()


### Calinski–Harabasz Index

The Calinski–Harabasz index is defined as:

$$
\text{CH} = 
\frac{\text{Tr}(B_k)}{\text{Tr}(W_k)} 
\cdot
\frac{N - k}{k - 1}
$$

where:
- $B_k$ is the between-cluster dispersion matrix,
- $W_k$ is the within-cluster dispersion matrix,
- $N$ is the number of samples,
- $k$ is the number of clusters.

**Higher** values $\longrightarrow$ **better**-defined clusters.

This score can be calculated through `calinski_harabasz_score` in `sklearn.metrics`.

In [ ]:
# HERE COMES CODE: Import calinski_harabasz_score
ch_scores = []

for k in range(2, 10):
    model = None # HERE COMES CODE: Define K-means with k number clusters. Set random_state to a value
    labels = None # HERE COMES CODE: Use fit_predict to predict labels
    ch = None # HERE COMES CODE: Calculate calinski_harabasz_score
    ch_scores.append(ch)

plt.figure(figsize=(7,5))
plt.plot(K, ch_scores, 'o-', color='darkorange')
plt.title("Calinski–Harabasz Score vs Number of Clusters")
plt.xlabel("Number of clusters (k)")
plt.ylabel("CH score")
plt.grid(True)
plt.show()


### Davies–Bouldin Index

The Davies–Bouldin index measures how similar each cluster is to its most similar other cluster.

It is defined as:

$$
DB = \frac{1}{k} \sum_{i=1}^{k} 
\max_{j \neq i} 
\frac{s_i + s_j}{d_{ij}}
$$

where:
- $s_i$ is the average intra-cluster distance for cluster $i$,  
- $d_{ij}$ is the distance between the centroids of clusters $i$ and $j$.

**Lower** values $\longrightarrow$ **better** clustering.

This score can be calculated through `davies_bouldin_score` in `sklearn.metrics`.

In [ ]:
# HERE COMES CODE: Import davies_bouldin_score

db_scores = []

for k in range(2, 10):
    model = None # HERE COMES CODE: Define K-means with k number clusters. Set random_state to a value
    labels = None # HERE COMES CODE: Use fit_predict to predict labels
    db = None # HERE COMES CODE: Calculate davies_bouldin_score
    db_scores.append(db)

plt.figure(figsize=(7,5))
plt.plot(K, db_scores, 'o-', color='seagreen')
plt.title("Davies–Bouldin Score vs Number of Clusters")
plt.xlabel("Number of clusters (k)")
plt.ylabel("DB score (lower is better)")
plt.grid(True)
plt.show()


## 3- External validation

External validation refers to the situation in which we evaluate how well the clustering output matches some known labels.

External validation measures **agreement** between two partitions:

1. the predicted clustering, and
2. another set of labels.

These metrics do not tell us whether the clusters are meaningful in an unsupervised sense — they only tell us how similar they are to some other labelling of the data.

What labels can we compare with?

- Labels that represent categories we expect clustering to recover. Examples include benchmark datasets, simulations where the true classes are known, or real applications where some prior knowledge exists.
- Labels obtained from a different clustering algorithm (to compare methods).
- Labels obtained by applying the same clustering algorithm to variations of the data (for example, subsampling or adding noise), in order to study stability.

### Adjusted Rand Index (ARI)

The Rand Index (RI) counts how many *pairs* of points are assigned:

- to the same cluster in both partitions, or  
- to different clusters in both partitions.

The **Adjusted** Rand Index corrects RI for chance agreement.

The ARI is defined as:

$$
\text{ARI} =
\frac{
\text{RI} - \text{Expected RI}
}{
\text{Max RI} - \text{Expected RI}
}.
$$

Properties:

- ARI ranges from $-1$ to $1$  
- $1$ → perfect match  
- $0$ → no better than random  
- $<0$ → worse than random  
- ARI is invariant to label permutation (cluster 0 vs 1 does not matter)


This score can be found through `adjusted_rand_score` in `sklearn.metrics`. 

In [ ]:
# HERE COMES CODE: Import adjusted_rand_score

model = None # HERE COMES CODE: define KMeans with n_clusters=3 and a random_state
labels = None # HERE COMES CODE: retrieve labels

# HERE COMES CODE: Calculate AIR

### Normalized Mutual Information (NMI)

Mutual Information (MI) measures how much knowing one label partition reduces 
uncertainty about the other.

The **Normalized** MI scales MI to the range $0$ to $1$:

$$
\text{NMI}(C, T) =
\frac{
2 \, I(C, T)
}{
H(C) + H(T)
}
$$

where:

- $I(C,T)$ is mutual information between cluster labels $C$ and true labels $T$,
- $H(C)$ and $H(T)$ are their entropies.

Properties:

- NMI ranges from $0$ to $1$  
- $1$ → perfect agreement  
- $0$ → no shared information  
- Invariant to label permutations

NMI is more tolerant than ARI when clusters partly overlap.

This score can be found through `normalized_mutual_info_score` in `sklearn.metrics`. 


In [ ]:
# HERE COMES CODE: Import normalized_mutual_info_score
model = None # HERE COMES CODE: define KMeans with n_clusters=3 and a random_state
labels = None # HERE COMES CODE: retrieve labels
# HERE COMES CODE: Calculate NMI

<div class="alert alert-block alert-info">
    💡💡<b>REMEMBER:</b> External validation measures <b>agreement</b>, not intrinsic quality.💡💡
</div>

### 4- Stability-Based Cluster Validation

Stability-based cluster validation involves assessing whether clusters are meaningful or not. 

To this end, one checks how *stable* they are when we slightly perturb the data.

The underlying idea is simple:

- If clusters reflect real structure, they should be reproducible  across different subsamples or different runs of the algorithm.

- If clusters are unstable, they are likely capturing noise, initialization, artifacts, or sampling variability.

Stability is therefore a measure of robustness and reliability, not just geometric compactness.


### Subsampling Stability

Subsampling stability measures how robust a clustering solution is when we 
remove a portion of the data. The idea is that if clusters capture real 
structure, they should be reproducible across different random subsamples.

Procedure:
1. Draw a random subsample of the data (e.g., 80%).
2. Apply the clustering algorithm and store the labels.
3. Draw another independent subsample.
4. Cluster it again.
5. Compare the two sets of cluster labels **only on the points that appear in both subsamples**, using an agreement measure such as ARI or NMI.
6. Repeat several times and average the agreement.

High average agreement → stable clustering.  
Low agreement → unstable clustering, likely capturing noise or sampling variability.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import numpy as np

n_samples = X.shape[0]
size = int(0.8 * n_samples)   # 80% subsample size
n_runs = 30
aris_sub = []

for _ in range(n_runs):
    # First subsample
    idx1 = None # CODE COMES HERE: use choice from random with n_samples, size=size and replace=False
    X1 = X[idx1]
    labels1 = None # CODE COMES HERE: Define KMeans with 3 clusters, fit and predict on X1 to generate cluster assignments
    
    # Second subsample
    idx2 = None # CODE COMES HERE: use choice from random with n_samples, size=size and replace=False 
    X2 = X[idx2]
    labels2 = None # CODE COMES HERE: Define KMeans with 3 clusters, fit and predict on X2 to generate cluster assignments
    
    # Overlapping points between subsamples
    common = None # CODE COMES HERE: use intersect1d in numpy on idx1 and idx2
    
    # Map to local indices in each subsample
    map1 = [np.where(idx1 == i)[0][0] for i in common]
    map2 = [np.where(idx2 == i)[0][0] for i in common]
    
    # ARI between the two clusterings
    ari = # CODE COMES HERE: Calculate ARI on the predicted labels of subsample1 and subsample2
    aris_sub.append(ari)

np.mean(aris_sub)

### Bootstrap Stability

Bootstrap stability is similar to subsampling, but subsamples are drawn 
**with replacement**. This means some points may appear multiple times in the 
same bootstrap sample, while others may not appear at all.

This method evaluates how sensitive clustering is to:
- duplicated influential points,
- small changes in the sample composition,
- outliers or points with high leverage.

Procedure:
1. Draw a bootstrap sample (same size as the dataset, but sampled with replacement).
2. Cluster it.
3. Draw another bootstrap sample and cluster again.
4. Compare the two clusterings using ARI or NMI on the points they share.
5. Repeat multiple times.

Bootstrap stability is useful when the dataset is small or when you want to 
model the effect of repeated observations.


In [ ]:
n_samples = X.shape[0]
n_runs = 30
aris_boot = []

for _ in range(n_runs):
    # Bootstrap samples
    idx1 = # CODE COMES HERE: use choice from random with n_samples, size=n_samples and replace=True
    idx2 = # CODE COMES HERE: use choice from random with n_samples, size=n_samples and replace=True
    
    labels1 = KMeans(n_clusters=3).fit_predict(X[idx1])
    labels2 = KMeans(n_clusters=3).fit_predict(X[idx2])
    
    # Overlapping indices
    common = np.intersect1d(idx1, idx2)
    
    # Map to each bootstrap’s local index
    map1 = [np.where(idx1 == i)[0][0] for i in common]
    map2 = [np.where(idx2 == i)[0][0] for i in common]
    
    ari = adjusted_rand_score(labels1[map1], labels2[map2])
    aris_boot.append(ari)

np.mean(aris_boot)


### Split-Half Stability

Split-half stability tests whether clustering results replicate across two 
independent subsets of the data. This method is conceptually similar to 
cross-validation in supervised learning.

Procedure:
1. Randomly split the dataset into two halves.
2. Cluster each half independently.
3. Compare the two clusterings on the full data, or by using a classifier to map clusters from one half to the other.
4. Repeat the split several times and compute the average agreement (ARI or NMI).

If the clustering solution is meaningful, both halves of the data should 
produce compatible cluster structures. Large inconsistencies indicate that 
clusters depend heavily on specific samples and may not reflect stable patterns.


#### Approach 1

Compare predicted labels of the full data from both clustering models.

In [ ]:
n_samples = X.shape[0]
n_runs = 30
aris_half = []

for _ in range(n_runs):
    # Random shuffle
    perm = None # HERE COMES CODE: Perfume idxs 
    half = None # HERE COMES CODE: Calculate half point

    idx1 = None # HERE COMES CODE: Select first half of indices 
    idx2 = None # HERE COMES CODE: Select second half of indices 
    
    # Cluster each half independently
    model_1 = None # HERE COMES CODE: fitted KMeans with three clusters on first half of data 
    model_2 = None # HERE COMES CODE: fitted KMeans with three clusters on second half of data 
    
    # Predict clusters for ALL samples using each half-model
    labels_1 = None # HERE COMES CODE: predict labels using model 1 on full data
    labels_2 = None # HERE COMES CODE: predict labels using model 2 on full data
    
    ari = None # HERE COMES CODE: Calculate ARI using labels_1 and labels_2
    aris_half.append(ari)

np.mean(aris_half)

#### Approach 2 

Use clustering assignments from one partition to train a classifier, generate labels on the second partition and compare these with its clustering solution.

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np

n_samples = X.shape[0]
n_runs = 30
aris_half_classifier = []

for _ in range(n_runs):
    # Random shuffle
    perm = None # HERE COMES CODE: Perfume idxs 
    half = None # HERE COMES CODE: Calculate half point

    idx1 = None # HERE COMES CODE: Select first half of indices 
    idx2 = None # HERE COMES CODE: Select second half of indices 
    
    # Cluster each half independently
    model_1 = None # HERE COMES CODE: fitted KMeans with three clusters on first half of data 
    model_2 = None # HERE COMES CODE: fitted KMeans with three clusters on second half of data 
    
    # Predict clusters for ALL samples using each half-model
    labels_1 = None # HERE COMES CODE: predict labels using model 1 on first half of data
    labels_2 = None # HERE COMES CODE: predict labels using model 2 on second half of data
    
    clf = None # HERE COMES CODE: Define Logistic Regression object 
    # HERE COMES CODE: Fit to first half
    
    # Predict cluster labels on half 2
    labels2_pred = None # HERE COMES CODE: Predict on second half

    # Compare through ARI
    ari = None  # HERE COMES CODE: calculate ARI between Kmeans labels and predicte labels from model in 1
    aris_half_classifier.append(ari)

np.mean(aris_half_classifier)

## 5. Exercises

### 5.1 Clustering and Validation on the Wine Dataset

Using the Wine dataset:

1. Apply the following algorithms with 3 clusters and store their labels:  
   K-Means, Gaussian Mixture Models, and Agglomerative Clustering (Ward linkage).

2. Visualize each clustering solution using any of the approaches covered (e.g., PCA, MDS, distance matrix, heatmap).

3. Perform internal validation using any metric(s) discussed in class (e.g., Silhouette, Calinski–Harabasz).

4. Perform external validation by comparing each clustering solution with the true labels.

5. Assess the stability of the clustering solutions using any of the stability approaches introduced.  
   You may focus on just one clustering algorithm for this part.


In [ ]:
from sklearn.datasets import load_wine
data = load_wine()
X = data.data
y = data.target

### 5.2. Clustering and Validation on the Iris Dataset

Repeat the same steps as is the previous exercise, but now using the **Iris** dataset instead. 

As you work through the steps, notice how clustering behaves differently on this dataset, and observe that the task is more challenging. Why do you think this is it?

In [ ]:
from sklearn.datasets import load_wine
wine = load_wine()
X = wine.data
y = wine.target